# NB02 — Swiss-Prot Candidate Assembly

Build the candidate enzyme list for each gap-fill reaction using Rosetta's tiered mappings.
For reactions with no Rosetta evidence, use reaction_similarity as a proxy.
Fetch protein sequences from refdata_uniprot.

**Requires**: BERDL Spark session (on-cluster or proxy)

**Inputs**: `data/gapfill_landscape.csv`, Rosetta parquet files

**Outputs**: `data/swissprot_candidates.parquet`, `data/proxy_candidates.parquet`, `data/all_candidates.parquet`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

from berdl_notebook_utils.setup_spark_session import get_spark_session
spark = get_spark_session()

ROSETTA = Path('../..') / 'rosetta' / 'data'
USER_DATA = Path('..') / 'user_data'
DATA_OUT = Path('..') / 'user_data'
DATA_OUT.mkdir(exist_ok=True)

print('Spark session ready.')

Spark session ready.


## 1. Load Gap-Fill Landscape from NB01

In [2]:
landscape = pd.read_csv(DATA_OUT / 'gapfill_landscape.csv')
print(f"Total gap-fill reactions: {len(landscape)}")

has_sp = set(landscape[landscape['n_swissprot'] > 0]['rxn'])
no_evidence = set(landscape[landscape['confidence'].isin(['none', 'missing'])]['rxn'])
evidence_no_sp = set(landscape[(landscape['n_swissprot'] == 0) & (~landscape['confidence'].isin(['none', 'missing']))]['rxn'])

print(f"With Swiss-Prot candidates: {len(has_sp)}")
print(f"No Rosetta evidence (need proxy): {len(no_evidence)}")
print(f"Evidence but no Swiss-Prot: {len(evidence_no_sp)}")

Total gap-fill reactions: 42
With Swiss-Prot candidates: 25
No Rosetta evidence (need proxy): 16
Evidence but no Swiss-Prot: 1


## 2. Build Direct Swiss-Prot Candidate List

In [3]:
ec_rxn = pd.read_parquet(ROSETTA / 'ec_to_reaction.parquet')
protein_ec = pd.read_parquet(ROSETTA / 'uniprot_native_protein_ec.parquet')
swissprot = pd.read_parquet(ROSETTA / 'swissprot_proteins.parquet')
sp_ids = set(swissprot.iloc[:, 0])

gf_ec = ec_rxn[ec_rxn['rxn_bare'].isin(has_sp)][['rxn_bare', 'ec']]
gf_ecs = set(gf_ec['ec'])
sp_protein_ec = protein_ec[protein_ec['ec'].isin(gf_ecs) & protein_ec['protein'].isin(sp_ids)]

direct_candidates = (
    gf_ec
    .merge(sp_protein_ec[['protein', 'ec']], on='ec')
    .rename(columns={'rxn_bare': 'rxn', 'protein': 'uniprot_id'})
    [['rxn', 'uniprot_id', 'ec']]
    .drop_duplicates()
)
direct_candidates['method'] = 'direct'
direct_candidates['proxy_rxn'] = None
direct_candidates['proxy_similarity'] = None

print(f"Direct candidates: {len(direct_candidates)} (reaction, protein) pairs")
print(f"Unique proteins: {direct_candidates['uniprot_id'].nunique()}")
print(f"Reactions covered: {direct_candidates['rxn'].nunique()}")

Direct candidates: 1308 (reaction, protein) pairs
Unique proteins: 1140
Reactions covered: 25


## 3. Proxy Search for No-Evidence Reactions

For reactions with no Rosetta evidence, find similar reactions via `reaction_similarity`
that DO have Swiss-Prot candidates.

In [ ]:
proxy_rxns = no_evidence | evidence_no_sp
print(f"Reactions needing proxy search: {len(proxy_rxns)}")
print(f"Reactions: {sorted(proxy_rxns)}")

all_rosetta_rxns_with_sp = set(ec_rxn['rxn_bare'])

proxy_records = []
for rxn in sorted(proxy_rxns):
    db_rxn_id = f"seed.reaction:{rxn}"
    query = f"""
        SELECT reaction_2, similarity
        FROM kbase_msd_biochemistry.reaction_similarity
        WHERE reaction_1 = '{db_rxn_id}' AND CAST(similarity AS FLOAT) > 0.7
        ORDER BY CAST(similarity AS FLOAT) DESC
        LIMIT 20
    """
    result = spark.sql(query).toPandas()
    
    if result.empty:
        print(f"  {rxn}: no similar reactions found")
        continue
    
    result['sim_rxn_bare'] = result['reaction_2'].str.replace('seed.reaction:', '', regex=False)
    result_with_sp = result[result['sim_rxn_bare'].isin(all_rosetta_rxns_with_sp)]
    
    if result_with_sp.empty:
        print(f"  {rxn}: {len(result)} similar, but none have Swiss-Prot links")
        continue
    
    for _, row in result_with_sp.iterrows():
        sim_rxn = row['sim_rxn_bare']
        sim_score = float(row['similarity'])
        
        sim_ecs = set(ec_rxn[ec_rxn['rxn_bare'] == sim_rxn]['ec'])
        sim_sp = protein_ec[protein_ec['ec'].isin(sim_ecs) & protein_ec['protein'].isin(sp_ids)]
        
        for uid in sim_sp['protein'].unique():
            ec_val = sim_sp[sim_sp['protein'] == uid]['ec'].iloc[0]
            proxy_records.append({
                'rxn': rxn,
                'uniprot_id': uid,
                'ec': ec_val,
                'method': 'proxy',
                'proxy_rxn': sim_rxn,
                'proxy_similarity': sim_score
            })
    
    n_proxies = len([r for r in proxy_records if r['rxn'] == rxn])
    print(f"  {rxn}: {len(result_with_sp)} similar rxns with SP links -> {n_proxies} candidates")

proxy_df = pd.DataFrame(proxy_records).drop_duplicates(subset=['rxn', 'uniprot_id'])
print(f"\nProxy candidates: {len(proxy_df)} pairs")
print(f"Unique proxy proteins: {proxy_df['uniprot_id'].nunique() if len(proxy_df) > 0 else 0}")
print(f"Reactions covered by proxy: {proxy_df['rxn'].nunique() if len(proxy_df) > 0 else 0}")

## 4. Combine All Candidates

In [ ]:
frames = [df for df in [direct_candidates, proxy_df] if not df.empty]
all_candidates = pd.concat(frames, ignore_index=True)
all_candidates = all_candidates.drop_duplicates(subset=['rxn', 'uniprot_id'])

print(f"Total candidates: {len(all_candidates)}")
print(f"Unique proteins: {all_candidates['uniprot_id'].nunique()}")
print(f"Reactions covered: {all_candidates['rxn'].nunique()} / {len(landscape)}")
print(f"\nBy method:")
print(all_candidates['method'].value_counts().to_string())

uncovered = set(landscape['rxn']) - set(all_candidates['rxn'])
if uncovered:
    print(f"\nStill uncovered ({len(uncovered)}): {sorted(uncovered)}")

## 5. Fetch Protein Sequences from BERDL

In [6]:
unique_proteins = sorted(all_candidates['uniprot_id'].unique())
print(f"Fetching sequences for {len(unique_proteins)} unique proteins...")

db_ids = [f"uniprot:{uid}" for uid in unique_proteins]

batch_size = 500
seq_records = []
for i in range(0, len(db_ids), batch_size):
    batch = db_ids[i:i+batch_size]
    id_list = ", ".join([f"'{x}'" for x in batch])
    query = f"""
        SELECT protein_id, sequence
        FROM refdata_uniprot.protein
        WHERE protein_id IN ({id_list})
    """
    result = spark.sql(query).toPandas()
    seq_records.append(result)
    print(f"  Batch {i//batch_size + 1}: {len(result)} sequences")

seq_df = pd.concat(seq_records, ignore_index=True)
seq_df['uniprot_id'] = seq_df['protein_id'].str.replace('uniprot:', '', regex=False)
seq_df = seq_df[['uniprot_id', 'sequence']]

print(f"\nSequences retrieved: {len(seq_df)} / {len(unique_proteins)}")

missing_seqs = set(unique_proteins) - set(seq_df['uniprot_id'])
if missing_seqs:
    print(f"Missing from refdata_uniprot: {len(missing_seqs)}")
    print(f"  Examples: {list(missing_seqs)[:5]}")

Fetching sequences for 3168 unique proteins...


  Batch 1: 500 sequences


  Batch 2: 500 sequences


  Batch 3: 500 sequences


  Batch 4: 500 sequences


  Batch 5: 500 sequences


  Batch 6: 500 sequences


  Batch 7: 168 sequences

Sequences retrieved: 3168 / 3168


In [7]:
if missing_seqs:
    print("Checking user_data/uniprot-msd-links.tsv for missing sequences...")
    fallback_seqs = {}
    with open(USER_DATA / 'uniprot-msd-links.tsv') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3 and parts[0] in missing_seqs:
                fallback_seqs[parts[0]] = parts[2]
    
    if fallback_seqs:
        fallback_df = pd.DataFrame([
            {'uniprot_id': k, 'sequence': v} for k, v in fallback_seqs.items()
        ])
        seq_df = pd.concat([seq_df, fallback_df], ignore_index=True)
        print(f"  Recovered {len(fallback_seqs)} sequences from fallback")
    
    still_missing = missing_seqs - set(fallback_seqs.keys())
    if still_missing:
        print(f"  Still missing (will be excluded): {len(still_missing)}")
else:
    print("All sequences found in refdata_uniprot.")

All sequences found in refdata_uniprot.


## 6. Merge and Save

In [8]:
all_candidates = all_candidates.merge(seq_df, on='uniprot_id', how='inner')

print(f"Final candidate table: {len(all_candidates)} rows")
print(f"Unique proteins with sequences: {all_candidates['uniprot_id'].nunique()}")
print(f"Reactions covered: {all_candidates['rxn'].nunique()}")
print(f"\nSequence length stats:")
all_candidates['seq_len'] = all_candidates['sequence'].str.len()
print(all_candidates['seq_len'].describe().to_string())

print(f"\nCandidates per reaction:")
per_rxn = all_candidates.groupby('rxn')['uniprot_id'].nunique().sort_values(ascending=False)
print(per_rxn.to_string())

Final candidate table: 4123 rows
Unique proteins with sequences: 3168
Reactions covered: 32

Sequence length stats:
count    4123.000000
mean      405.891099
std       193.807425
min        15.000000
25%       275.500000
50%       357.000000
75%       493.000000
max      2753.000000

Candidates per reaction:
rxn
rxn12298    1126
rxn04455     929
rxn04660     369
rxn01327     368
rxn12225     142
rxn13647     140
rxn01361      99
rxn02107      98
rxn04658      96
rxn03978      79
rxn08491      79
rxn01737      76
rxn00575      73
rxn04659      71
rxn01927      57
rxn00738      54
rxn05119      53
rxn05117      53
rxn04656      36
rxn00128      28
rxn02227      14
rxn02226      14
rxn00141      13
rxn01398      12
rxn05902      12
rxn00510       8
rxn00317       6
rxn00084       6
rxn01663       4
rxn00577       4
rxn03883       3
rxn04657       1


In [9]:
direct_out = all_candidates[all_candidates['method'] == 'direct']
proxy_out = all_candidates[all_candidates['method'] == 'proxy']

direct_out.to_parquet(DATA_OUT / 'swissprot_candidates.parquet', index=False)
if len(proxy_out) > 0:
    proxy_out.to_parquet(DATA_OUT / 'proxy_candidates.parquet', index=False)
all_candidates.to_parquet(DATA_OUT / 'all_candidates.parquet', index=False)

print(f"Saved {len(direct_out)} direct candidates to data/swissprot_candidates.parquet")
print(f"Saved {len(proxy_out)} proxy candidates to data/proxy_candidates.parquet")
print(f"Saved {len(all_candidates)} total candidates to data/all_candidates.parquet")
print(f"\nReady for NB03: fetch ESM-2 embeddings for {all_candidates['uniprot_id'].nunique()} unique proteins.")

Saved 1303 direct candidates to data/swissprot_candidates.parquet
Saved 2820 proxy candidates to data/proxy_candidates.parquet
Saved 4123 total candidates to data/all_candidates.parquet

Ready for NB03: fetch ESM-2 embeddings for 3168 unique proteins.
